# 05. Commitment Dynamics

This notebook combines the archived sliding-window table with the scalar `time_to_commit` metric from `unified_metrics.csv`.
        


In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

ROOT = Path.cwd().resolve()
if not (ROOT / 'data').exists():
    ROOT = ROOT.parent
DATA = ROOT / 'data'
FIGURES = ROOT / 'figures'
FIGURES.mkdir(exist_ok=True)
sys.path.insert(0, str(ROOT))

from src.statistical_analysis import cohens_d, two_way_anova, stratified_logistic_auc

sns.set_theme(style='whitegrid', font_scale=1.1)
pd.set_option('display.max_columns', 120)

window_df = pd.read_csv(DATA / 'analysis' / 'analysis_C_window_metrics.csv')
unified = pd.read_csv(DATA / 'qwen05b' / 'unified_metrics.csv')
window_df.head()
        


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
summary = window_df.groupby(['t_start', 'group'])['window_rg'].agg(['mean', 'sem']).reset_index()
for group in ['G1', 'G2', 'G3', 'G4']:
    sub = summary[summary['group'] == group]
    axes[0].plot(sub['t_start'], sub['mean'], label=group)
    axes[0].fill_between(sub['t_start'], sub['mean'] - sub['sem'], sub['mean'] + sub['sem'], alpha=0.15)
axes[0].set_title('Sliding-window radius of gyration')
axes[0].set_xlabel('Window start token')
axes[0].set_ylabel('Windowed $R_g$')
axes[0].legend()

layer0 = unified[unified['layer'] == 0].copy()
sns.kdeplot(data=layer0, x='time_to_commit', hue='condition', fill=True, common_norm=False, ax=axes[1])
axes[1].set_title('Distribution of time-to-commit by regime')
axes[1].set_xlabel('time_to_commit')
plt.tight_layout()
plt.savefig(FIGURES / 'repro_05_commitment_dynamics.png', dpi=300)
plt.show()

summary_stats = layer0.groupby('condition')['time_to_commit'].agg(['mean', 'median', 'std', 'count'])
summary_stats
        
